In [1]:
# Cell 1: Install dependencies

!pip3 install groq pandas numpy sentence-transformers transformers torch scikit-learn -q


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# Cell 2: Imports

import os
import re
import time
import math
import getpass
import numpy as np
import pandas as pd
import torch

from typing import Dict, Any, List
from collections import Counter
from groq import Groq
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

/Users/soumyajitbera/Documents/GitHub/Token_monitoring_and_optimisation/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Cell 3: Groq Client Setup

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter GROQ_API_KEY: ")

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

MODEL = "llama-3.3-70b-versatile"

In [4]:
# Cell 4: Load Local Models

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
PPL_MODEL_NAME = "distilgpt2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

ppl_tokenizer = AutoTokenizer.from_pretrained(PPL_MODEL_NAME)
ppl_model = AutoModelForCausalLM.from_pretrained(PPL_MODEL_NAME)
ppl_model.eval()

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 17183.28it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [5]:
# Cell 5: Core Utility Functions

def approx_tokens(text: str) -> int:
    return max(1, math.ceil(len(text) / 4))


def lexical_entropy(text: str) -> float:
    words = re.findall(r"\b\w+\b", text.lower())
    if not words:
        return 0.0

    counts = Counter(words)
    total = len(words)

    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def redundancy_ratio(text: str) -> float:
    units = [u.strip().lower() for u in re.split(r"[\n\.]", text) if u.strip()]
    if not units:
        return 0.0

    return 1 - (len(set(units)) / len(units))


def information_density(text: str) -> float:
    return lexical_entropy(text) / approx_tokens(text)

In [6]:
# Cell 6: Text Normalization

FILLER_TERMS = [
    "please",
    "kindly",
    "can you",
    "could you",
    "i want",
    "i need",
    "properly",
    "end to end",
    "don't miss anything",
    "as per above",
    "give me",
    "now",
    "well",
    "basically",
    "like"
]


def normalize_text(text: str) -> str:
    text = text.strip()
    text = text.replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)

    for term in FILLER_TERMS:
        text = re.sub(rf"\b{re.escape(term)}\b", "", text, flags=re.IGNORECASE)

    text = re.sub(r" {2,}", " ", text)
    return text.strip()

In [7]:
# Cell 7: Sentence Split + Embedding Similarity

def sentence_split(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+|\n+", text)
    return [p.strip() for p in parts if p.strip()]


def embedding_cosine_similarity(text_a: str, text_b: str) -> float:
    embeddings = embedding_model.encode([text_a, text_b], convert_to_numpy=True)

    a = embeddings[0]
    b = embeddings[1]

    denom = np.linalg.norm(a) * np.linalg.norm(b)

    if denom == 0:
        return 0.0

    return round(float(np.dot(a, b) / denom), 4)


def cosine_similarity_np(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)

    if denom == 0:
        return 0.0

    return float(np.dot(a, b) / denom)

In [8]:
# Cell 8: Embedding-Based Semantic Pruning

def embedding_semantic_prune(
    text: str,
    query: str,
    similarity_threshold: float = 0.88
) -> str:
    spans = sentence_split(text)

    if len(spans) <= 2:
        return text

    span_embeddings = embedding_model.encode(spans, convert_to_numpy=True)
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0]

    kept_spans = []
    kept_embeddings = []

    for span, emb in zip(spans, span_embeddings):
        duplicate = False

        for kept_emb in kept_embeddings:
            sim = cosine_similarity_np(emb, kept_emb)

            if sim >= similarity_threshold:
                duplicate = True
                break

        if not duplicate:
            kept_spans.append(span)
            kept_embeddings.append(emb)

    if len(kept_spans) > 6:
        relevance_scores = [
            cosine_similarity_np(emb, query_embedding)
            for emb in kept_embeddings
        ]

        cutoff = np.percentile(relevance_scores, 25)

        selected = {
            span
            for span, emb in zip(kept_spans, kept_embeddings)
            if cosine_similarity_np(emb, query_embedding) >= cutoff
        }

        kept_spans = [span for span in kept_spans if span in selected]

    return "\n".join(kept_spans)

In [9]:
# Cell 9: Perplexity + Structural Signal

def calculate_perplexity(text: str, max_length: int = 512) -> float:
    if not text.strip():
        return 1.0

    inputs = ppl_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    with torch.no_grad():
        outputs = ppl_model(
            **inputs,
            labels=inputs["input_ids"]
        )

    loss = outputs.loss.item()
    ppl = math.exp(min(loss, 20))

    return round(float(ppl), 4)


def structural_signal_score(text: str) -> float:
    patterns = [
        r"```",
        r"\bdef\b",
        r"\bclass\b",
        r"\bimport\b",
        r"\btraceback\b",
        r"\berror\b",
        r"\bexception\b",
        r"\bjson\b",
        r"\bapi\b",
        r"\bsql\b",
        r"\bfunction\b",
        r"\{.*\}",
        r"\[.*\]"
    ]

    count = 0

    for pattern in patterns:
        if re.search(pattern, text.lower(), flags=re.DOTALL):
            count += 1

    return min(count / len(patterns), 1.0)

In [10]:
# Cell 10: Advanced Complexity Score

def advanced_complexity_score(query: str) -> Dict[str, Any]:
    token_len = approx_tokens(query)
    ppl = calculate_perplexity(query)
    structural = structural_signal_score(query)
    red = redundancy_ratio(query)

    length_score = min(token_len / 1200, 1.0)
    ppl_score = min(math.log1p(ppl) / 8.0, 1.0)

    score = (
        0.30 * length_score
        + 0.40 * ppl_score
        + 0.40 * structural
        - 0.20 * red
    )

    score = max(0.05, min(score, 1.0))

    return {
        "advanced_complexity_score": round(score, 4),
        "perplexity": ppl,
        "ppl_score": round(ppl_score, 4),
        "structural_signal": round(structural, 4),
        "length_score": round(length_score, 4),
        "redundancy_ratio": round(red, 4),
    }

In [11]:
# Cell 11: Semantic Quality + Preservation

def semantic_quality_score(answer: str, query: str) -> float:
    semantic_sim = embedding_cosine_similarity(query, answer)

    query_terms = set(re.findall(r"\w+", query.lower()))
    answer_terms = set(re.findall(r"\w+", answer.lower()))

    lexical_coverage = len(query_terms & answer_terms) / max(len(query_terms), 1)
    repetition_penalty = redundancy_ratio(answer)

    quality = (
        0.55 * semantic_sim
        + 0.30 * lexical_coverage
        + 0.15 * (1 - repetition_penalty)
    )

    return round(max(0.0, min(quality, 1.0)), 4)


def preservation_score(original_query: str, optimized_query: str) -> float:
    return embedding_cosine_similarity(original_query, optimized_query)

In [12]:
# Cell 12: Aggressive Optimizer

def sentence_deduplicate_jaccard(text: str, threshold: float = 0.82) -> str:
    sentences = sentence_split(text)
    kept = []

    for sent in sentences:
        sent_words = set(re.findall(r"\w+", sent.lower()))
        duplicate = False

        for old in kept:
            old_words = set(re.findall(r"\w+", old.lower()))
            if not sent_words or not old_words:
                continue

            sim = len(sent_words & old_words) / len(sent_words | old_words)

            if sim >= threshold:
                duplicate = True
                break

        if not duplicate:
            kept.append(sent)

    return "\n".join(kept)


def allocate_output_budget_v1(query: str) -> int:
    q = query.lower()
    c = min(approx_tokens(query) / 1000, 1.0)

    base_min = 120
    base_max = 620

    if any(x in q for x in ["summarize", "summary", "compress"]):
        base_min, base_max = 90, 220

    if any(x in q for x in ["json", "extract", "schema", "fields"]):
        base_min, base_max = 120, 300

    if any(x in q for x in ["error", "bug", "fix", "traceback"]):
        base_min, base_max = 140, 360

    if any(x in q for x in ["linkedin", "creative", "post"]):
        base_min, base_max = 180, 450

    return int(base_min + (base_max - base_min) * c)


def optimize_input_v1(query: str) -> Dict[str, Any]:
    original = query

    q1 = normalize_text(query)
    q2 = sentence_deduplicate_jaccard(q1)

    q_best = q2 if approx_tokens(q2) <= approx_tokens(q1) else q1

    original_tokens = approx_tokens(original)
    optimized_tokens = approx_tokens(q_best)

    return {
        "optimizer": "v1",
        "original_query": original,
        "optimized_query": q_best,
        "original_est_tokens": original_tokens,
        "optimized_est_tokens": optimized_tokens,
        "estimated_input_reduction_percent": round(
            ((original_tokens - optimized_tokens) / max(original_tokens, 1)) * 100,
            2
        ),
        "complexity_score": round(min(approx_tokens(q_best) / 1000, 1.0), 4),
        "adaptive_output_budget": allocate_output_budget_v1(q_best),
        "information_density": round(information_density(q_best), 6),
    }

In [13]:
# Cell 13:  Semantic Safety Optimizer

def regression_budget_predictor_simple(
    query: str,
    budget_min: int = 100,
    budget_max: int = 520
) -> int:
    adv = advanced_complexity_score(query)
    c = adv["advanced_complexity_score"]

    budget = budget_min + int((budget_max - budget_min) * c)
    return int(max(budget_min, min(budget, budget_max)))


def optimize_input_v2_1(
    query: str,
    similarity_threshold: float = 0.88,
    preservation_threshold: float = 0.90,
    budget_min: int = 100,
    budget_max: int = 520
) -> Dict[str, Any]:

    original = query

    q1 = normalize_text(query)

    q2 = embedding_semantic_prune(
        q1,
        q1,
        similarity_threshold=similarity_threshold
    )

    preserve = preservation_score(q1, q2)

    if preserve < preservation_threshold:
        q_best = q1
        compression_accepted = False
    else:
        q_best = q2
        compression_accepted = True

    code_markers = [
        "traceback",
        "error",
        "exception",
        "code:",
        "def ",
        "class ",
        "import "
    ]

    if any(m in original.lower() for m in code_markers):
        if approx_tokens(q_best) < approx_tokens(original) * 0.70:
            q_best = q1
            compression_accepted = False

    adv = advanced_complexity_score(q_best)

    budget = regression_budget_predictor_simple(
        query=q_best,
        budget_min=budget_min,
        budget_max=budget_max
    )

    original_tokens = approx_tokens(original)
    optimized_tokens = approx_tokens(q_best)

    return {
        "optimizer": "v21",
        "original_query": original,
        "optimized_query": q_best,
        "original_est_tokens": original_tokens,
        "optimized_est_tokens": optimized_tokens,
        "estimated_input_reduction_percent": round(
            ((original_tokens - optimized_tokens) / max(original_tokens, 1)) * 100,
            2
        ),
        "preservation_score": preserve,
        "compression_accepted": compression_accepted,
        "complexity_score": adv["advanced_complexity_score"],
        "perplexity": adv["perplexity"],
        "ppl_score": adv["ppl_score"],
        "structural_signal": adv["structural_signal"],
        "adaptive_output_budget": budget,
        "information_density": round(information_density(q_best), 6),
    }

In [14]:
# Cell 14: Minimal Contract + Stop Sequences

def build_minimal_contract() -> str:
    return (
        "Answer compactly. "
        "No intro. No conclusion. "
        "Use only required words/code/data. "
        "Do not explain unless needed."
    )


STOP_SEQUENCES = [
    "\nConclusion:",
    "\nAdditional Notes:",
    "\nReferences:",
    "\nHope this helps"
]

In [15]:
# Cell 15: Groq Calls

def ask_groq_baseline(query: str, model: str = MODEL) -> Dict[str, Any]:
    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": query
            }
        ],
        temperature=0.7,
        top_p=1.0,
        max_completion_tokens=1200,
    )

    usage = response.usage

    return {
        "optimizer": "baseline",
        "answer": response.choices[0].message.content.strip(),
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_seconds": round(time.time() - start, 3),
    }


def call_groq_with_optimized_input(
    optimized: Dict[str, Any],
    model: str = MODEL,
    top_p: float = 0.60,
    temperature: float = 0.0
) -> Dict[str, Any]:

    messages = [
        {
            "role": "system",
            "content": build_minimal_contract()
        },
        {
            "role": "user",
            "content": optimized["optimized_query"]
        }
    ]

    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        top_p=top_p,
        max_completion_tokens=optimized["adaptive_output_budget"],
        stop=STOP_SEQUENCES,
        seed=42,
    )

    usage = response.usage
    answer = response.choices[0].message.content.strip()

    return {
        **optimized,
        "answer": answer,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_seconds": round(time.time() - start, 3),
    }


def ask_groq_v1(query: str, model: str = MODEL) -> Dict[str, Any]:
    optimized = optimize_input_v1(query)
    return call_groq_with_optimized_input(optimized, model=model)


def ask_groq_v21(query: str, model: str = MODEL) -> Dict[str, Any]:
    optimized = optimize_input_v2_1(query)
    return call_groq_with_optimized_input(optimized, model=model)

In [16]:
# Cell 16:  Hybrid Routing Features

def query_routing_features(query: str) -> Dict[str, Any]:
    q = query.lower()

    adv = advanced_complexity_score(query)

    token_len = approx_tokens(query)
    struct = adv["structural_signal"]
    ppl_score = adv["ppl_score"]
    complexity = adv["advanced_complexity_score"]
    red = redundancy_ratio(query)

    code_debug_signal = int(any(x in q for x in [
        "error",
        "exception",
        "traceback",
        "bug",
        "fix",
        "code:",
        "def ",
        "class ",
        "import "
    ]))

    extraction_signal = int(any(x in q for x in [
        "json",
        "extract",
        "schema",
        "fields",
        "parse"
    ]))

    summarization_signal = int(any(x in q for x in [
        "summarize",
        "summary",
        "compress"
    ]))

    creative_signal = int(any(x in q for x in [
        "linkedin",
        "creative",
        "post",
        "story",
        "caption"
    ]))

    return {
        "token_len": token_len,
        "complexity_score": complexity,
        "ppl_score": ppl_score,
        "structural_signal": struct,
        "redundancy_ratio": red,
        "code_debug_signal": code_debug_signal,
        "extraction_signal": extraction_signal,
        "summarization_signal": summarization_signal,
        "creative_signal": creative_signal,
    }

In [17]:
# Cell 17:  Hybrid Router

RISK_THRESHOLD = 0.30


def choose_optimizer_v3_1(query: str) -> Dict[str, Any]:
    f = query_routing_features(query)

    semantic_risk_score = (
        0.35 * f["complexity_score"]
        + 0.25 * f["structural_signal"]
        + 0.20 * f["ppl_score"]
        + 0.20 * f["code_debug_signal"]
    )

    compression_opportunity_score = (
        0.45 * f["redundancy_ratio"]
        + 0.25 * min(f["token_len"] / 600, 1.0)
        + 0.15 * f["summarization_signal"]
        + 0.15 * f["creative_signal"]
    )

    selected = "v1"
    reason = "Cost-first path selected."

    if semantic_risk_score >= RISK_THRESHOLD:
        selected = "v21"
        reason = "Semantic risk exceeds tuned threshold; V2.1 selected."

    return {
        **f,
        "semantic_risk_score": round(semantic_risk_score, 4),
        "compression_opportunity_score": round(compression_opportunity_score, 4),
        "selected_optimizer": selected,
        "routing_reason": reason,
        "risk_threshold": RISK_THRESHOLD,
    }


def ask_groq_v3_1(query: str, model: str = MODEL) -> Dict[str, Any]:
    route = choose_optimizer_v3_1(query)

    if route["selected_optimizer"] == "v21":
        result = ask_groq_v21(query=query, model=model)
    else:
        result = ask_groq_v1(query=query, model=model)

    result["v31_selected_optimizer"] = route["selected_optimizer"]
    result["v31_routing_reason"] = route["routing_reason"]
    result["semantic_risk_score"] = route["semantic_risk_score"]
    result["compression_opportunity_score"] = route["compression_opportunity_score"]
    result["risk_threshold"] = route["risk_threshold"]

    return result

In [18]:
# Cell 18: Cost and Evaluation Metrics

INPUT_PRICE_PER_1M = 0.59
OUTPUT_PRICE_PER_1M = 0.79


def cost_usd(input_tokens: int, output_tokens: int) -> float:
    return (
        (input_tokens / 1_000_000) * INPUT_PRICE_PER_1M
        + (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    )


def compression_factor(original_tokens: int, optimized_tokens: int) -> float:
    if optimized_tokens <= 0:
        return 1.0

    return round(original_tokens / optimized_tokens, 4)


def token_efficiency_score(
    information_density_value: float,
    quality_score: float,
    total_tokens: int
) -> float:
    if total_tokens <= 0:
        return 0.0

    return round(
        (information_density_value * quality_score) / total_tokens,
        8
    )


def normalize_metric(value: float, min_value: float, max_value: float) -> float:
    if max_value == min_value:
        return 0.0

    return round(
        (value - min_value) / (max_value - min_value),
        6
    )


def global_optimization_efficiency_score(
    tes_norm: float,
    cost_reduction_norm: float,
    quality_score: float,
    compression_norm: float
) -> float:
    score = (
        0.35 * tes_norm
        + 0.25 * cost_reduction_norm
        + 0.20 * quality_score
        + 0.20 * compression_norm
    )

    return round(score, 6)

In [19]:
# Cell 19: Test Queries

test_queries = {
    "factual_qa": """
What is Retrieval Augmented Generation? Explain with one simple example.
""",

    "code_generation": """
Write a Python function that calls Groq API and returns the answer along with
input tokens, output tokens, and total tokens.
""",

    "summarization": """
Summarize this:

ProdSync is an AI-powered SaaS platform for talent discovery, candidate evaluation,
and hiring intelligence. It helps job seekers improve employability using ATS resume
analysis, skill-gap detection, AI role recommendation, JD-resume matching, GitHub
project audit, readiness scoring, and AI mock interviews. For recruiters, it provides
semantic candidate search, AI shortlisting, automated first-round interview analysis,
and hireability reports. The platform targets Tier-2 and Tier-3 college students,
placement cells, and hiring teams in India.
""",

    "debugging": """
I am getting this Python error:

TypeError: unsupported operand type(s) for +: 'int' and 'str'

Code:
age = 25
message = "My age is " + age
print(message)

Find the bug and give the smallest correct fix.
""",

    "json_extraction": """
Extract the following information as JSON:

Candidate Name: Soumyajit Bera
Current Role: AI Engineer
Company: IBM
Experience: 2 years 9 months
Skills: Python, FastAPI, LangChain, Milvus, SQL, Machine Learning
Expected CTC: 25 LPA
Preferred Location: Kolkata
""",

    "creative": """
Write a powerful LinkedIn post announcing ProdSync as an AI-powered employability
and hiring intelligence platform for Tier-2 and Tier-3 college students in India.
Keep it founder-style, confident, and inspiring.
"""
}

In [20]:
# Cell 20: Single Query Test

query = test_queries["debugging"]

baseline = ask_groq_baseline(query)
v1 = ask_groq_v1(query)
v21 = ask_groq_v21(query)
v31 = ask_groq_v3_1(query)

print("Baseline Tokens:", baseline["total_tokens"])
print("V1 Tokens      :", v1["total_tokens"])
print("V2.1 Tokens    :", v21["total_tokens"])
print("V3.1 Tokens    :", v31["total_tokens"])

print("\nV3.1 Selected Optimizer:", v31["v31_selected_optimizer"])
print("Routing Reason:", v31["v31_routing_reason"])

print("\n========== V3.1 ANSWER ==========\n")
print(v31["answer"])

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Baseline Tokens: 213
V1 Tokens      : 158
V2.1 Tokens    : 124
V3.1 Tokens    : 124

V3.1 Selected Optimizer: v21
Routing Reason: Semantic risk exceeds tuned threshold; V2.1 selected.

========== V3.1 ANSWER ==========

```python
age = 25
message = "My age is " + str(age)
print(message)
```


In [21]:
# Cell 21: Batch Evaluation

rows = []

for use_case, query in test_queries.items():
    print(f"Running: {use_case}")

    baseline = ask_groq_baseline(query)
    v1 = ask_groq_v1(query)
    v21 = ask_groq_v21(query)
    v31 = ask_groq_v3_1(query)

    baseline_cost = cost_usd(baseline["input_tokens"], baseline["output_tokens"])
    v1_cost = cost_usd(v1["input_tokens"], v1["output_tokens"])
    v21_cost = cost_usd(v21["input_tokens"], v21["output_tokens"])
    v31_cost = cost_usd(v31["input_tokens"], v31["output_tokens"])

    v1_quality = semantic_quality_score(v1["answer"], query)
    v21_quality = semantic_quality_score(v21["answer"], query)
    v31_quality = semantic_quality_score(v31["answer"], query)

    cf = compression_factor(
        v31["original_est_tokens"],
        v31["optimized_est_tokens"]
    )

    tes = token_efficiency_score(
        information_density_value=v31["information_density"],
        quality_score=v31_quality,
        total_tokens=v31["total_tokens"]
    )

    rows.append({
        "use_case": use_case,

        "selected_optimizer": v31["v31_selected_optimizer"],
        "routing_reason": v31["v31_routing_reason"],

        "baseline_total_tokens": baseline["total_tokens"],
        "v1_total_tokens": v1["total_tokens"],
        "v21_total_tokens": v21["total_tokens"],
        "v31_total_tokens": v31["total_tokens"],

        "v1_cost_reduction_percent": round(((baseline_cost - v1_cost) / baseline_cost) * 100, 2),
        "v21_cost_reduction_percent": round(((baseline_cost - v21_cost) / baseline_cost) * 100, 2),
        "v31_cost_reduction_percent": round(((baseline_cost - v31_cost) / baseline_cost) * 100, 2),

        "v1_quality": v1_quality,
        "v21_quality": v21_quality,
        "v31_quality": v31_quality,

        "v31_token_reduction_percent": round(
            ((baseline["total_tokens"] - v31["total_tokens"]) / baseline["total_tokens"]) * 100,
            2
        ),

        "semantic_risk_score": v31["semantic_risk_score"],
        "compression_opportunity_score": v31["compression_opportunity_score"],

        "v31_tes": tes,
        "v31_compression_factor": cf,
        "v31_information_density": v31["information_density"],
    })

results_df = pd.DataFrame(rows)
results_df

Running: factual_qa
Running: code_generation
Running: summarization
Running: debugging
Running: json_extraction
Running: creative


,use_case,selected_optimizer,routing_reason,baseline_total_tokens,v1_total_tokens,v21_total_tokens,v31_total_tokens,v1_cost_reduction_percent,v21_cost_reduction_percent,v31_cost_reduction_percent,v1_quality,v21_quality,v31_quality,v31_token_reduction_percent,semantic_risk_score,compression_opportunity_score,v31_tes,v31_compression_factor,v31_information_density
0,factual_qa,v1,Cost-first path selected.,307,162,167,148,51.28,49.50,56.03,0.7231,0.6913,0.7176,51.79,0.1989,0.0079,0.000895,1.0556,0.184552
1,code_generation,v1,Cost-first path selected.,694,219,246,219,70.86,66.88,70.86,0.5650,0.5765,0.5776,68.44,0.2780,0.0133,0.000346,1.0323,0.131311
2,summarization,v1,Cost-first path selected.,222,246,241,246,-9.82,-10.13,-9.82,0.8118,0.6132,0.8118,-10.81,0.2087,0.2092,0.000137,1.0071,0.041511
3,debugging,v21,Semantic risk exceeds tuned threshold; V2.1 se...,203,158,124,124,28.16,45.32,45.32,0.7014,0.6764,0.6764,38.92,0.4073,0.0217,0.000634,1.4054,0.116311
4,json_extraction,v1,Cost-first path selected.,207,220,189,220,-4.11,10.52,-4.11,0.8497,0.7653,0.8497,-6.28,0.2541,0.0271,0.000312,1.0156,0.080780
5,creative,v1,Cost-first path selected.,574,184,184,184,71.38,71.38,71.38,0.7543,0.7543,0.7543,67.94,0.2218,0.1725,0.000374,1.0189,0.091315


In [22]:
# Cell 22: Add GOES

def add_goes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["tes_norm"] = df["v31_tes"].apply(
        lambda x: normalize_metric(
            x,
            df["v31_tes"].min(),
            df["v31_tes"].max()
        )
    )

    df["cost_reduction_norm"] = df["v31_cost_reduction_percent"].apply(
        lambda x: normalize_metric(
            x,
            df["v31_cost_reduction_percent"].min(),
            df["v31_cost_reduction_percent"].max()
        )
    )

    df["compression_norm"] = df["v31_compression_factor"].apply(
        lambda x: normalize_metric(
            x,
            df["v31_compression_factor"].min(),
            df["v31_compression_factor"].max()
        )
    )

    df["v31_goes"] = df.apply(
        lambda r: global_optimization_efficiency_score(
            tes_norm=r["tes_norm"],
            cost_reduction_norm=r["cost_reduction_norm"],
            quality_score=r["v31_quality"],
            compression_norm=r["compression_norm"]
        ),
        axis=1
    )

    return df


results_df = add_goes(results_df)
results_df.sort_values("v31_goes", ascending=False)

,use_case,selected_optimizer,routing_reason,baseline_total_tokens,v1_total_tokens,v21_total_tokens,v31_total_tokens,v1_cost_reduction_percent,v21_cost_reduction_percent,v31_cost_reduction_percent,...,v31_token_reduction_percent,semantic_risk_score,compression_opportunity_score,v31_tes,v31_compression_factor,v31_information_density,tes_norm,cost_reduction_norm,compression_norm,v31_goes
3,debugging,v21,Semantic risk exceeds tuned threshold; V2.1 se...,203,158,124,124,28.16,45.32,45.32,...,38.92,0.4073,0.0217,0.000634,1.4054,0.116311,0.656431,0.679064,1.000000,0.734797
0,factual_qa,v1,Cost-first path selected.,307,162,167,148,51.28,49.50,56.03,...,51.79,0.1989,0.0079,0.000895,1.0556,0.184552,1.000000,0.810961,0.121768,0.720614
5,creative,v1,Cost-first path selected.,574,184,184,184,71.38,71.38,71.38,...,67.94,0.2218,0.1725,0.000374,1.0189,0.091315,0.313193,1.000000,0.029626,0.516403
1,code_generation,v1,Cost-first path selected.,694,219,246,219,70.86,66.88,70.86,...,68.44,0.2780,0.0133,0.000346,1.0323,0.131311,0.276232,0.993596,0.063269,0.473254
4,json_extraction,v1,Cost-first path selected.,207,220,189,220,-4.11,10.52,-4.11,...,-6.28,0.2541,0.0271,0.000312,1.0156,0.080780,0.230919,0.070320,0.021341,0.272610
2,summarization,v1,Cost-first path selected.,222,246,241,246,-9.82,-10.13,-9.82,...,-10.81,0.2087,0.2092,0.000137,1.0071,0.041511,0.000000,0.000000,0.000000,0.162360


In [23]:
# Cell 23: Final Summary

summary = {
    "avg_v1_cost_reduction_percent": round(results_df["v1_cost_reduction_percent"].mean(), 2),
    "avg_v21_cost_reduction_percent": round(results_df["v21_cost_reduction_percent"].mean(), 2),
    "avg_v31_cost_reduction_percent": round(results_df["v31_cost_reduction_percent"].mean(), 2),

    "avg_v1_quality": round(results_df["v1_quality"].mean(), 4),
    "avg_v21_quality": round(results_df["v21_quality"].mean(), 4),
    "avg_v31_quality": round(results_df["v31_quality"].mean(), 4),

    "avg_v31_token_reduction_percent": round(results_df["v31_token_reduction_percent"].mean(), 2),
    "avg_v31_goes": round(results_df["v31_goes"].mean(), 4),

    "v31_selected_v1_count": int((results_df["selected_optimizer"] == "v1").sum()),
    "v31_selected_v21_count": int((results_df["selected_optimizer"] == "v21").sum()),
}

summary

{'avg_v1_cost_reduction_percent': np.float64(34.62),
 'avg_v21_cost_reduction_percent': np.float64(38.91),
 'avg_v31_cost_reduction_percent': np.float64(38.28),
 'avg_v1_quality': np.float64(0.7342),
 'avg_v21_quality': np.float64(0.6795),
 'avg_v31_quality': np.float64(0.7312),
 'avg_v31_token_reduction_percent': np.float64(35.0),
 'avg_v31_goes': np.float64(0.48),
 'v31_selected_v1_count': 5,
 'v31_selected_v21_count': 1}

In [24]:
# Cell 24: Human-Readable Final Interpretation

print("="*80)
print("FINAL V3.1 HYBRID TOKEN OPTIMIZER ANALYSIS")
print("="*80)

print("\nCost Reduction:")
print(f"V1   : {summary['avg_v1_cost_reduction_percent']}%")
print(f"V2.1 : {summary['avg_v21_cost_reduction_percent']}%")
print(f"V3.1 : {summary['avg_v31_cost_reduction_percent']}%")

print("\nQuality:")
print(f"V1   : {summary['avg_v1_quality']}")
print(f"V2.1 : {summary['avg_v21_quality']}")
print(f"V3.1 : {summary['avg_v31_quality']}")

print("\nToken Reduction:")
print(f"V3.1 : {summary['avg_v31_token_reduction_percent']}%")

print("\nGOES:")
print(f"V3.1 : {summary['avg_v31_goes']}")

print("\nRouting Distribution:")
print(f"V1 selected   : {summary['v31_selected_v1_count']} cases")
print(f"V2.1 selected : {summary['v31_selected_v21_count']} cases")

print("\nVerdict:")

if (
    summary["avg_v31_cost_reduction_percent"] >= summary["avg_v1_cost_reduction_percent"]
    and summary["avg_v31_quality"] >= summary["avg_v1_quality"]
):
    print("✅ V3.1 beats V1 and is production-ready.")

elif (
    summary["avg_v31_quality"] >= summary["avg_v1_quality"]
    and summary["avg_v31_cost_reduction_percent"] >= summary["avg_v1_cost_reduction_percent"] - 3
):
    print("⚠️ V3.1 is a balanced production candidate.")

else:
    print("❌ V3.1 still needs more routing/budget tuning.")

FINAL V3.1 HYBRID TOKEN OPTIMIZER ANALYSIS

Cost Reduction:
V1   : 34.62%
V2.1 : 38.91%
V3.1 : 38.28%

Quality:
V1   : 0.7342
V2.1 : 0.6795
V3.1 : 0.7312

Token Reduction:
V3.1 : 35.0%

GOES:
V3.1 : 0.48

Routing Distribution:
V1 selected   : 5 cases
V2.1 selected : 1 cases

Verdict:
❌ V3.1 still needs more routing/budget tuning.


### Tuned Process

In [25]:
# Cell 1: Install dependencies

!pip3 install groq pandas numpy sentence-transformers transformers torch -q


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [26]:
# Cell 2: Imports

import os
import re
import time
import math
import getpass
import numpy as np
import pandas as pd
import torch

from typing import Dict, Any, List
from collections import Counter
from groq import Groq
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [27]:
# Cell 3: Groq Setup

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter GROQ_API_KEY: ")

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

MODEL = "llama-3.3-70b-versatile"

In [28]:
# Cell 4: Load Local Models

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

ppl_tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
ppl_model = AutoModelForCausalLM.from_pretrained("distilgpt2")
ppl_model.eval()

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 15019.18it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [29]:
# Cell 5: Core Utilities

def approx_tokens(text: str) -> int:
    return max(1, math.ceil(len(text) / 4))


def lexical_entropy(text: str) -> float:
    words = re.findall(r"\b\w+\b", text.lower())
    if not words:
        return 0.0

    counts = Counter(words)
    total = len(words)

    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def redundancy_ratio(text: str) -> float:
    units = [u.strip().lower() for u in re.split(r"[\n\.]", text) if u.strip()]
    if not units:
        return 0.0

    return 1 - (len(set(units)) / len(units))


def information_density(text: str) -> float:
    return lexical_entropy(text) / approx_tokens(text)


def sentence_split(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+|\n+", text)
    return [p.strip() for p in parts if p.strip()]

In [30]:
# Cell 6: Normalization

FILLER_TERMS = [
    "please", "kindly", "can you", "could you",
    "i want", "i need", "properly", "end to end",
    "don't miss anything", "as per above", "give me",
    "now", "well", "basically", "like"
]


def normalize_text(text: str) -> str:
    text = text.strip()
    text = text.replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)

    for term in FILLER_TERMS:
        text = re.sub(rf"\b{re.escape(term)}\b", "", text, flags=re.IGNORECASE)

    return re.sub(r" {2,}", " ", text).strip()

In [31]:
# Cell 7: Embedding Similarity

def embedding_cosine_similarity(text_a: str, text_b: str) -> float:
    emb = embedding_model.encode([text_a, text_b], convert_to_numpy=True)

    a, b = emb[0], emb[1]
    denom = np.linalg.norm(a) * np.linalg.norm(b)

    if denom == 0:
        return 0.0

    return round(float(np.dot(a, b) / denom), 4)


def cosine_similarity_np(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)

    if denom == 0:
        return 0.0

    return float(np.dot(a, b) / denom)

In [32]:
# Cell 8: Embedding Semantic Pruning

def embedding_semantic_prune(
    text: str,
    query: str,
    similarity_threshold: float = 0.88
) -> str:

    spans = sentence_split(text)

    if len(spans) <= 2:
        return text

    span_embeddings = embedding_model.encode(spans, convert_to_numpy=True)
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0]

    kept_spans = []
    kept_embeddings = []

    for span, emb in zip(spans, span_embeddings):
        duplicate = False

        for kept_emb in kept_embeddings:
            if cosine_similarity_np(emb, kept_emb) >= similarity_threshold:
                duplicate = True
                break

        if not duplicate:
            kept_spans.append(span)
            kept_embeddings.append(emb)

    if len(kept_spans) > 6:
        relevance_scores = [
            cosine_similarity_np(emb, query_embedding)
            for emb in kept_embeddings
        ]

        cutoff = np.percentile(relevance_scores, 25)

        selected = {
            span
            for span, emb in zip(kept_spans, kept_embeddings)
            if cosine_similarity_np(emb, query_embedding) >= cutoff
        }

        kept_spans = [span for span in kept_spans if span in selected]

    return "\n".join(kept_spans)

In [33]:
# Cell 9: Perplexity and Structural Signals

def calculate_perplexity(text: str, max_length: int = 512) -> float:
    if not text.strip():
        return 1.0

    inputs = ppl_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    with torch.no_grad():
        outputs = ppl_model(
            **inputs,
            labels=inputs["input_ids"]
        )

    loss = outputs.loss.item()
    return round(float(math.exp(min(loss, 20))), 4)


def structural_signal_score(text: str) -> float:
    patterns = [
        r"```", r"\bdef\b", r"\bclass\b", r"\bimport\b",
        r"\btraceback\b", r"\berror\b", r"\bexception\b",
        r"\bjson\b", r"\bapi\b", r"\bsql\b", r"\bfunction\b",
        r"\{.*\}", r"\[.*\]"
    ]

    count = 0

    for pattern in patterns:
        if re.search(pattern, text.lower(), flags=re.DOTALL):
            count += 1

    return min(count / len(patterns), 1.0)

In [34]:
# Cell 10: Advanced Complexity Score

def advanced_complexity_score(query: str) -> Dict[str, Any]:
    token_len = approx_tokens(query)
    ppl = calculate_perplexity(query)
    structural = structural_signal_score(query)
    red = redundancy_ratio(query)

    length_score = min(token_len / 1200, 1.0)
    ppl_score = min(math.log1p(ppl) / 8.0, 1.0)

    score = (
        0.30 * length_score
        + 0.40 * ppl_score
        + 0.40 * structural
        - 0.20 * red
    )

    score = max(0.05, min(score, 1.0))

    return {
        "advanced_complexity_score": round(score, 4),
        "perplexity": ppl,
        "ppl_score": round(ppl_score, 4),
        "structural_signal": round(structural, 4),
        "length_score": round(length_score, 4),
        "redundancy_ratio": round(red, 4),
    }

In [35]:
# Cell 11: Semantic Quality and Preservation

def semantic_quality_score(answer: str, query: str) -> float:
    semantic_sim = embedding_cosine_similarity(query, answer)

    query_terms = set(re.findall(r"\w+", query.lower()))
    answer_terms = set(re.findall(r"\w+", answer.lower()))

    lexical_coverage = len(query_terms & answer_terms) / max(len(query_terms), 1)
    repetition_penalty = redundancy_ratio(answer)

    quality = (
        0.55 * semantic_sim
        + 0.30 * lexical_coverage
        + 0.15 * (1 - repetition_penalty)
    )

    return round(max(0.0, min(quality, 1.0)), 4)


def preservation_score(original_query: str, optimized_query: str) -> float:
    return embedding_cosine_similarity(original_query, optimized_query)

In [36]:
# Cell 12: V1 Aggressive Cost Optimizer

def sentence_deduplicate_jaccard(text: str, threshold: float = 0.82) -> str:
    sentences = sentence_split(text)
    kept = []

    for sent in sentences:
        sent_words = set(re.findall(r"\w+", sent.lower()))
        duplicate = False

        for old in kept:
            old_words = set(re.findall(r"\w+", old.lower()))

            if not sent_words or not old_words:
                continue

            sim = len(sent_words & old_words) / len(sent_words | old_words)

            if sim >= threshold:
                duplicate = True
                break

        if not duplicate:
            kept.append(sent)

    return "\n".join(kept)


def allocate_output_budget_v1(query: str) -> int:
    q = query.lower()
    c = min(approx_tokens(query) / 1000, 1.0)

    base_min, base_max = 120, 620

    if any(x in q for x in ["summarize", "summary", "compress"]):
        base_min, base_max = 90, 220

    if any(x in q for x in ["json", "extract", "schema", "fields"]):
        base_min, base_max = 120, 300

    if any(x in q for x in ["error", "bug", "fix", "traceback"]):
        base_min, base_max = 140, 360

    if any(x in q for x in ["linkedin", "creative", "post"]):
        base_min, base_max = 180, 450

    return int(base_min + (base_max - base_min) * c)


def optimize_input_v1(query: str) -> Dict[str, Any]:
    original = query

    q1 = normalize_text(query)
    q2 = sentence_deduplicate_jaccard(q1)

    q_best = q2 if approx_tokens(q2) <= approx_tokens(q1) else q1

    return {
        "optimizer": "v1",
        "original_query": original,
        "optimized_query": q_best,
        "original_est_tokens": approx_tokens(original),
        "optimized_est_tokens": approx_tokens(q_best),
        "complexity_score": round(min(approx_tokens(q_best) / 1000, 1.0), 4),
        "adaptive_output_budget": allocate_output_budget_v1(q_best),
        "information_density": round(information_density(q_best), 6),
    }

In [37]:
# Cell 13: V2.1 Semantic-Safe Optimizer

def regression_budget_predictor_simple(
    query: str,
    budget_min: int = 100,
    budget_max: int = 520
) -> int:
    adv = advanced_complexity_score(query)
    c = adv["advanced_complexity_score"]

    return int(budget_min + (budget_max - budget_min) * c)


def optimize_input_v2_1(
    query: str,
    similarity_threshold: float = 0.88,
    preservation_threshold: float = 0.90,
    budget_min: int = 100,
    budget_max: int = 520
) -> Dict[str, Any]:

    original = query

    q1 = normalize_text(query)
    q2 = embedding_semantic_prune(
        q1,
        q1,
        similarity_threshold=similarity_threshold
    )

    preserve = preservation_score(q1, q2)

    if preserve < preservation_threshold:
        q_best = q1
        compression_accepted = False
    else:
        q_best = q2
        compression_accepted = True

    code_markers = [
        "traceback", "error", "exception",
        "code:", "def ", "class ", "import "
    ]

    if any(m in original.lower() for m in code_markers):
        if approx_tokens(q_best) < approx_tokens(original) * 0.70:
            q_best = q1
            compression_accepted = False

    adv = advanced_complexity_score(q_best)

    return {
        "optimizer": "v21",
        "original_query": original,
        "optimized_query": q_best,
        "original_est_tokens": approx_tokens(original),
        "optimized_est_tokens": approx_tokens(q_best),
        "preservation_score": preserve,
        "compression_accepted": compression_accepted,
        "complexity_score": adv["advanced_complexity_score"],
        "perplexity": adv["perplexity"],
        "ppl_score": adv["ppl_score"],
        "structural_signal": adv["structural_signal"],
        "adaptive_output_budget": regression_budget_predictor_simple(
            q_best,
            budget_min=budget_min,
            budget_max=budget_max
        ),
        "information_density": round(information_density(q_best), 6),
    }

In [38]:
# Cell 14: Prompt Contract and Stop Sequences

def build_minimal_contract() -> str:
    return (
        "Answer compactly. "
        "No intro. No conclusion. "
        "Use only required words/code/data. "
        "Do not explain unless needed."
    )


STOP_SEQUENCES = [
    "\nConclusion:",
    "\nAdditional Notes:",
    "\nReferences:",
    "\nHope this helps"
]

In [39]:
# Cell 15: Groq Calls

def ask_groq_baseline(query: str, model: str = MODEL) -> Dict[str, Any]:
    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": query}
        ],
        temperature=0.7,
        top_p=1.0,
        max_completion_tokens=1200,
    )

    usage = response.usage

    return {
        "optimizer": "baseline",
        "answer": response.choices[0].message.content.strip(),
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_seconds": round(time.time() - start, 3),
    }


def call_groq_with_optimized_input(
    optimized: Dict[str, Any],
    model: str = MODEL,
    top_p: float = 0.60,
    temperature: float = 0.0
) -> Dict[str, Any]:

    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": build_minimal_contract()},
            {"role": "user", "content": optimized["optimized_query"]}
        ],
        temperature=temperature,
        top_p=top_p,
        max_completion_tokens=optimized["adaptive_output_budget"],
        stop=STOP_SEQUENCES,
        seed=42,
    )

    usage = response.usage

    return {
        **optimized,
        "answer": response.choices[0].message.content.strip(),
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_seconds": round(time.time() - start, 3),
    }


def ask_groq_v1(query: str, model: str = MODEL) -> Dict[str, Any]:
    return call_groq_with_optimized_input(
        optimize_input_v1(query),
        model=model
    )


def ask_groq_v21(query: str, model: str = MODEL) -> Dict[str, Any]:
    return call_groq_with_optimized_input(
        optimize_input_v2_1(query),
        model=model
    )

In [40]:
# Cell 16: V3.2 Hybrid Router Features

def query_routing_features(query: str) -> Dict[str, Any]:
    q = query.lower()
    adv = advanced_complexity_score(query)

    code_debug_signal = int(any(x in q for x in [
        "error", "exception", "traceback", "bug", "fix",
        "code:", "def ", "class ", "import "
    ]))

    extraction_signal = int(any(x in q for x in [
        "json", "extract", "schema", "fields", "parse"
    ]))

    summarization_signal = int(any(x in q for x in [
        "summarize", "summary", "compress"
    ]))

    creative_signal = int(any(x in q for x in [
        "linkedin", "creative", "post", "story", "caption"
    ]))

    return {
        "token_len": approx_tokens(query),
        "complexity_score": adv["advanced_complexity_score"],
        "ppl_score": adv["ppl_score"],
        "structural_signal": adv["structural_signal"],
        "redundancy_ratio": redundancy_ratio(query),
        "code_debug_signal": code_debug_signal,
        "extraction_signal": extraction_signal,
        "summarization_signal": summarization_signal,
        "creative_signal": creative_signal,
    }

In [41]:
# Cell 17: V3.2 Risk-vs-Compression Router

ROUTER_SCORE_THRESHOLD = 0.15


def choose_optimizer_v3_2(query: str) -> Dict[str, Any]:
    f = query_routing_features(query)

    semantic_risk_score = (
        0.35 * f["complexity_score"]
        + 0.25 * f["structural_signal"]
        + 0.20 * f["ppl_score"]
        + 0.20 * f["code_debug_signal"]
    )

    compression_opportunity_score = (
        0.45 * f["redundancy_ratio"]
        + 0.25 * min(f["token_len"] / 600, 1.0)
        + 0.15 * f["summarization_signal"]
        + 0.15 * f["creative_signal"]
    )

    router_score = (
        0.65 * semantic_risk_score
        - 0.35 * compression_opportunity_score
    )

    selected = "v1"
    reason = "Cost-first optimizer selected."

    if router_score >= ROUTER_SCORE_THRESHOLD:
        selected = "v21"
        reason = "Semantic risk outweighs compression opportunity; V2.1 selected."

    return {
        **f,
        "semantic_risk_score": round(semantic_risk_score, 4),
        "compression_opportunity_score": round(compression_opportunity_score, 4),
        "router_score": round(router_score, 4),
        "selected_optimizer": selected,
        "routing_reason": reason,
    }


def ask_groq_v3_2(query: str, model: str = MODEL) -> Dict[str, Any]:
    route = choose_optimizer_v3_2(query)

    if route["selected_optimizer"] == "v21":
        result = ask_groq_v21(query=query, model=model)
    else:
        result = ask_groq_v1(query=query, model=model)

    result["v32_selected_optimizer"] = route["selected_optimizer"]
    result["v32_routing_reason"] = route["routing_reason"]
    result["semantic_risk_score"] = route["semantic_risk_score"]
    result["compression_opportunity_score"] = route["compression_opportunity_score"]
    result["router_score"] = route["router_score"]

    return result

In [42]:
# Cell 18: Evaluation Metrics

INPUT_PRICE_PER_1M = 0.59
OUTPUT_PRICE_PER_1M = 0.79


def cost_usd(input_tokens: int, output_tokens: int) -> float:
    return (
        (input_tokens / 1_000_000) * INPUT_PRICE_PER_1M
        + (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    )


def compression_factor(original_tokens: int, optimized_tokens: int) -> float:
    if optimized_tokens <= 0:
        return 1.0

    return round(original_tokens / optimized_tokens, 4)


def token_efficiency_score(
    information_density_value: float,
    quality_score: float,
    total_tokens: int
) -> float:
    if total_tokens <= 0:
        return 0.0

    return round(
        (information_density_value * quality_score) / total_tokens,
        8
    )


def normalize_metric(value: float, min_value: float, max_value: float) -> float:
    if max_value == min_value:
        return 0.0

    return round((value - min_value) / (max_value - min_value), 6)


def global_optimization_efficiency_score(
    tes_norm: float,
    cost_reduction_norm: float,
    quality_score: float,
    compression_norm: float
) -> float:
    return round(
        0.35 * tes_norm
        + 0.25 * cost_reduction_norm
        + 0.20 * quality_score
        + 0.20 * compression_norm,
        6
    )

In [43]:
# Cell 19: Test Queries

test_queries = {
    "factual_qa": """
What is Retrieval Augmented Generation? Explain with one simple example.
""",

    "code_generation": """
Write a Python function that calls Groq API and returns the answer along with
input tokens, output tokens, and total tokens.
""",

    "summarization": """
Summarize this:

ProdSync is an AI-powered SaaS platform for talent discovery, candidate evaluation,
and hiring intelligence. It helps job seekers improve employability using ATS resume
analysis, skill-gap detection, AI role recommendation, JD-resume matching, GitHub
project audit, readiness scoring, and AI mock interviews. For recruiters, it provides
semantic candidate search, AI shortlisting, automated first-round interview analysis,
and hireability reports. The platform targets Tier-2 and Tier-3 college students,
placement cells, and hiring teams in India.
""",

    "debugging": """
I am getting this Python error:

TypeError: unsupported operand type(s) for +: 'int' and 'str'

Code:
age = 25
message = "My age is " + age
print(message)

Find the bug and give the smallest correct fix.
""",

    "json_extraction": """
Extract the following information as JSON:

Candidate Name: Soumyajit Bera
Current Role: AI Engineer
Company: IBM
Experience: 2 years 9 months
Skills: Python, FastAPI, LangChain, Milvus, SQL, Machine Learning
Expected CTC: 25 LPA
Preferred Location: Kolkata
""",

    "creative": """
Write a powerful LinkedIn post announcing ProdSync as an AI-powered employability
and hiring intelligence platform for Tier-2 and Tier-3 college students in India.
Keep it founder-style, confident, and inspiring.
"""
}

In [47]:
# Cell 20: Batch Evaluation

rows = []

for use_case, query in test_queries.items():
    print(f"Running: {use_case}")

    baseline = ask_groq_baseline(query)
    v1 = ask_groq_v1(query)
    v21 = ask_groq_v21(query)
    v32 = ask_groq_v3_2(query)

    baseline_cost = cost_usd(baseline["input_tokens"], baseline["output_tokens"])
    v1_cost = cost_usd(v1["input_tokens"], v1["output_tokens"])
    v21_cost = cost_usd(v21["input_tokens"], v21["output_tokens"])
    v32_cost = cost_usd(v32["input_tokens"], v32["output_tokens"])

    v1_quality = semantic_quality_score(v1["answer"], query)
    v21_quality = semantic_quality_score(v21["answer"], query)
    v32_quality = semantic_quality_score(v32["answer"], query)

    cf = compression_factor(
        v32["original_est_tokens"],
        v32["optimized_est_tokens"]
    )

    tes = token_efficiency_score(
        information_density_value=v32["information_density"],
        quality_score=v32_quality,
        total_tokens=v32["total_tokens"]
    )

    rows.append({
        "use_case": use_case,

        "selected_optimizer": v32["v32_selected_optimizer"],
        "routing_reason": v32["v32_routing_reason"],

        "baseline_total_tokens": baseline["total_tokens"],
        "v1_total_tokens": v1["total_tokens"],
        "v21_total_tokens": v21["total_tokens"],
        "v32_total_tokens": v32["total_tokens"],

        "v1_cost_reduction_percent": round(((baseline_cost - v1_cost) / baseline_cost) * 100, 2),
        "v21_cost_reduction_percent": round(((baseline_cost - v21_cost) / baseline_cost) * 100, 2),
        "v32_cost_reduction_percent": round(((baseline_cost - v32_cost) / baseline_cost) * 100, 2),

        "v1_quality": v1_quality,
        "v21_quality": v21_quality,
        "v32_quality": v32_quality,

        "v32_token_reduction_percent": round(
            ((baseline["total_tokens"] - v32["total_tokens"]) / baseline["total_tokens"]) * 100,
            2
        ),

        "semantic_risk_score": v32["semantic_risk_score"],
        "compression_opportunity_score": v32["compression_opportunity_score"],
        "router_score": v32["router_score"],

        "v32_tes": tes,
        "v32_compression_factor": cf,
        "v32_information_density": v32["information_density"],
    })

results_df = pd.DataFrame(rows)
results_df

Running: factual_qa


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jqzt8ryafhjtqye10djyrekx` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99014, Requested 1249. Please try again in 3m47.232s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# Cell 21: GOES Calculation

def add_goes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["tes_norm"] = df["v32_tes"].apply(
        lambda x: normalize_metric(x, df["v32_tes"].min(), df["v32_tes"].max())
    )

    df["cost_reduction_norm"] = df["v32_cost_reduction_percent"].apply(
        lambda x: normalize_metric(
            x,
            df["v32_cost_reduction_percent"].min(),
            df["v32_cost_reduction_percent"].max()
        )
    )

    df["compression_norm"] = df["v32_compression_factor"].apply(
        lambda x: normalize_metric(
            x,
            df["v32_compression_factor"].min(),
            df["v32_compression_factor"].max()
        )
    )

    df["v32_goes"] = df.apply(
        lambda r: global_optimization_efficiency_score(
            tes_norm=r["tes_norm"],
            cost_reduction_norm=r["cost_reduction_norm"],
            quality_score=r["v32_quality"],
            compression_norm=r["compression_norm"]
        ),
        axis=1
    )

    return df


results_df = add_goes(results_df)
results_df.sort_values("v32_goes", ascending=False)

In [ ]:
# Cell 22: Final Summary

summary = {
    "avg_v1_cost_reduction_percent": round(results_df["v1_cost_reduction_percent"].mean(), 2),
    "avg_v21_cost_reduction_percent": round(results_df["v21_cost_reduction_percent"].mean(), 2),
    "avg_v32_cost_reduction_percent": round(results_df["v32_cost_reduction_percent"].mean(), 2),

    "avg_v1_quality": round(results_df["v1_quality"].mean(), 4),
    "avg_v21_quality": round(results_df["v21_quality"].mean(), 4),
    "avg_v32_quality": round(results_df["v32_quality"].mean(), 4),

    "avg_v32_token_reduction_percent": round(results_df["v32_token_reduction_percent"].mean(), 2),
    "avg_v32_goes": round(results_df["v32_goes"].mean(), 4),

    "v32_selected_v1_count": int((results_df["selected_optimizer"] == "v1").sum()),
    "v32_selected_v21_count": int((results_df["selected_optimizer"] == "v21").sum()),
}

summary

In [ ]:
# Cell 23: Final Interpretation

print("="*80)
print("FINAL V3.2 HYBRID TOKEN OPTIMIZER ANALYSIS")
print("="*80)

print("\nCost Reduction:")
print(f"V1   : {summary['avg_v1_cost_reduction_percent']}%")
print(f"V2.1 : {summary['avg_v21_cost_reduction_percent']}%")
print(f"V3.2 : {summary['avg_v32_cost_reduction_percent']}%")

print("\nQuality:")
print(f"V1   : {summary['avg_v1_quality']}")
print(f"V2.1 : {summary['avg_v21_quality']}")
print(f"V3.2 : {summary['avg_v32_quality']}")

print("\nToken Reduction:")
print(f"V3.2 : {summary['avg_v32_token_reduction_percent']}%")

print("\nGOES:")
print(f"V3.2 : {summary['avg_v32_goes']}")

print("\nRouting Distribution:")
print(f"V1 selected   : {summary['v32_selected_v1_count']} cases")
print(f"V2.1 selected : {summary['v32_selected_v21_count']} cases")

print("\nVerdict:")

if (
    summary["avg_v32_cost_reduction_percent"] >= summary["avg_v1_cost_reduction_percent"]
    and summary["avg_v32_quality"] >= summary["avg_v1_quality"]
):
    print("✅ V3.2 beats V1 and is production-ready.")

elif (
    summary["avg_v32_cost_reduction_percent"] >= summary["avg_v1_cost_reduction_percent"]
    and summary["avg_v32_quality"] >= summary["avg_v1_quality"] - 0.01
):
    print("⚠️ V3.2 is production-viable: better cost with negligible quality loss.")

elif (
    summary["avg_v32_quality"] > summary["avg_v1_quality"]
    and summary["avg_v32_cost_reduction_percent"] >= summary["avg_v1_cost_reduction_percent"] - 3
):
    print("⚠️ V3.2 is a balanced production candidate.")

else:
    print("❌ V3.2 still needs routing/budget tuning.")